# Genes

This notebook shows the gene-level workflow with real `BioEmbedder.embed(...)` calls. The same gene list is embedded through DNA, protein, text, and morphology-backed models, annotated with embpy metadata utilities, and visualized with the plotting helpers.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import BioEmbedder, pl, tl

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


def short_embedding_key(key: str) -> str:
    parts = key.split("__")
    if len(parts) >= 3 and parts[0] == "X_emb":
        return f"X_{parts[2]}"
    return key


def feature_embeddings_as_obs(embedded: ad.AnnData, *, label_column: str = "label") -> ad.AnnData:
    """Convert real feature-aligned `.varm` embeddings to `.obsm` for plotting."""
    obs = embedded.var.copy()
    if label_column not in obs.columns:
        obs[label_column] = obs.index.astype(str)
    out = ad.AnnData(
        X=np.zeros((embedded.n_vars, 1), dtype=np.float32),
        obs=obs,
    )
    for key, matrix in embedded.varm.items():
        out.obsm[short_embedding_key(key)] = np.asarray(matrix, dtype=np.float32)
    out.uns["source_embeddings"] = {
        "varm_keys": list(embedded.varm.keys()),
        "note": "Matrices come directly from BioEmbedder.embed(..., output='anndata').",
    }
    return out

genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1"]
print(f"Genes: {genes}")


## Embed genes with DNA, protein, and text models

Gene and protein entities are feature-aligned in the scverse contract, so their matrices are stored in `.varm` on the returned AnnData. The helper below keeps those real matrices and mirrors them into `.obsm` only for plotting functions that expect observation-level embeddings.


In [ ]:
gene_embeddings = embedder.embed(
    genes,
    entity_type="gene",
    id_type="symbol",
    model=["hyenadna_tiny_1k", "esm2_8M", "minilm_l6_v2"],
    output="anndata",
    organism="human",
    pooling_strategy="mean",
    show_progress=True,
)

display(gene_embeddings)
print("varm keys:", list(gene_embeddings.varm.keys()))
print("embedding metadata keys:", list(gene_embeddings.uns["embeddings"].keys()))

gene_space = feature_embeddings_as_obs(gene_embeddings, label_column="gene_symbol")
if "gene_symbol" in gene_space.obs:
    gene_space.obs["symbol"] = gene_space.obs["gene_symbol"].fillna(
        pd.Series(gene_space.obs_names.astype(str), index=gene_space.obs_names)
    )
else:
    gene_space.obs["symbol"] = gene_space.obs_names.astype(str)

display(gene_space)
print("obsm keys for plotting:", list(gene_space.obsm.keys()))


## Add morphology perturbation embeddings

Perturbation/action embeddings are entity-aligned payloads in `.uns`. When a downstream model needs one vector per row, materialize the payload into `.obsm` explicitly.


In [ ]:
from embpy.io import materialize_perturbation_obsm

perturbation_adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"perturbation": genes}, index=[f"pert_{i}" for i in range(len(genes))]),
)

perturbation_adata = embedder.embed(
    perturbation_adata,
    entity_type="perturbation",
    obs_column="perturbation",
    model="subcell_mae_rybg",
    output="anndata",
    attach_to="uns",
    key="X_pert_subcell_mae_rybg",
    morphology_dataset="hpa",
    morphology_source="subcell",
    max_images=1,
    aggregation="mean",
    show_progress=True,
)
perturbation_adata = materialize_perturbation_obsm(
    perturbation_adata,
    embedding_key="X_pert_subcell_mae_rybg",
    perturbation_key="perturbation",
    obsm_key="X_pert_subcell_mae_rybg",
    missing="nan",
)

print("uns perturbation keys:", list(perturbation_adata.uns["perturbations"].keys()))
print("materialized shape:", perturbation_adata.obsm["X_pert_subcell_mae_rybg"].shape)


## Annotate genes and their protein products

Annotations are fetched through embpy's real metadata utilities and stored in `.obs` and `.uns`.


In [ ]:
gene_space = tl.annotate_gene_perturbations(
    gene_space,
    column="symbol",
    sources=["pathways", "interactions", "diseases"],
    copy=True,
)
gene_space = tl.annotate_proteins(
    gene_space,
    column="symbol",
    id_type="symbol",
    sources=["metadata", "location", "domains", "go", "interactions"],
    copy=True,
)

display(compact_obs(gene_space, ("gene_", "prot_"), base=["symbol", "gene_symbol"]))
print("annotation stores:", [k for k in gene_space.uns if "annotation" in k])


## Plot annotated embedding spaces


In [ ]:
color_key = "gene_n_pathways" if "gene_n_pathways" in gene_space.obs else "symbol"
pl.plot_embedding_space(
    gene_space,
    obsm_key="X_hyenadna_tiny_1k",
    method="pca",
    color=color_key,
    annotate=True,
    annotate_col="symbol",
    title="DNA gene embeddings colored by embpy gene annotations",
)

if "prot_location" in gene_space.obs:
    pl.plot_embedding_space(
        gene_space,
        obsm_key="X_esm2_8M",
        method="pca",
        color="prot_location",
        annotate=True,
        annotate_col="symbol",
        title="Protein-derived gene embeddings colored by UniProt location",
    )


## Compare embedding views


In [ ]:
k = min(3, gene_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(gene_space, "X_hyenadna_tiny_1k", "X_esm2_8M", k=k)
print(f"Mean DNA/protein KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(gene_space, obsm_keys=["X_hyenadna_tiny_1k", "X_esm2_8M", "X_minilm_l6_v2"], k=k)
pl.cross_embedding_correlation(gene_space, "X_hyenadna_tiny_1k", "X_esm2_8M")
pl.embedding_norms(gene_space, obsm_keys=["X_hyenadna_tiny_1k", "X_esm2_8M", "X_minilm_l6_v2"])


## Save a reusable artifact


In [ ]:
gene_embeddings.write_h5ad(OUTPUT_DIR / "gene_embeddings.h5ad")
print(OUTPUT_DIR / "gene_embeddings.h5ad")
